# EDA — Análisis Exploratorio de Matches y Features

Fase 2: Exploración de datos de partidos internacionales para el proyecto de predicción del Mundial 2026.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import DB_PATH

con = sqlite3.connect(DB_PATH)
matches = pd.read_sql("SELECT * FROM matches ORDER BY date", con, parse_dates=["date"])
teams = pd.read_sql("SELECT * FROM teams", con)
con.close()

print(f"Total de matches: {len(matches)}")
print(f"Rango de fechas: {matches['date'].min()} a {matches['date'].max()}")
print(f"Equipos únicos: {len(teams)}")

## Distribución de goles

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

matches["home_goals"].value_counts().sort_index().plot(kind="bar", ax=ax1, title="Goles Local")
matches["away_goals"].value_counts().sort_index().plot(kind="bar", ax=ax2, title="Goles Visitante")

plt.tight_layout()
plt.show()

print(f"Promedio de goles local: {matches['home_goals'].mean():.2f}")
print(f"Promedio de goles visitante: {matches['away_goals'].mean():.2f}")

## Resultados por tipo (1X2)

In [ ]:
matches["outcome"] = matches.apply(
    lambda r: "Home" if r["home_goals"] > r["away_goals"] 
             else ("Draw" if r["home_goals"] == r["away_goals"] else "Away"),
    axis=1
)

outcome_dist = matches["outcome"].value_counts()
print(outcome_dist)
print(f"\nProbabilidades: {outcome_dist / len(matches)}")

outcome_dist.plot(kind="bar", figsize=(8, 4), title="Distribución de Resultados")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

## Cobertura de datos por año

In [ ]:
matches["year"] = matches["date"].dt.year
matches_per_year = matches.groupby("year").size()

print(f"Partidos desde 2004: {len(matches[matches['year'] >= 2004])}")
print(f"Cobertura esperada (>=95%): {len(matches[matches['year'] >= 2004]) / len(matches) * 100:.1f}%")

matches_per_year[matches_per_year.index >= 2000].plot(kind="line", figsize=(12, 4), title="Matches por Año (desde 2000)")
plt.ylabel("Cantidad de partidos")
plt.tight_layout()
plt.show()

## Top 10 selecciones por número de partidos

In [ ]:
home_count = matches["home_team_id"].value_counts()
away_count = matches["away_team_id"].value_counts()
total_matches = home_count.add(away_count, fill_value=0)

team_names = dict(zip(teams["team_id"], teams["name"]))
top_teams = total_matches.nlargest(10)

print("Top 10 selecciones por partidos jugados:")
for tid, count in top_teams.items():
    print(f"  {team_names[tid]}: {int(count)}")

## Conclusiones preliminares

- Promedio de goles ligeramente mayor para local (ventaja local)  
- Distribución de goles compatible con Poisson  
- Cobertura de datos muy buena desde 2004 en adelante  
- Equipos tradicionales (Brasil, Alemania, Italia, Argentina, Francia) tienen excelente histórico de partidos
